In [1]:
# imports
import pandas as pd
import openpyxl
import numpy as np
import os
from collections import defaultdict
from openpyxl import load_workbook
import re

In [2]:
# pd.set_option('display.max_rows', None)

In [2]:
path = '/Users/emma/projects/cognoscolab/rassereno/data/fall2024_data collection'

## Functions and lists for free association and valence task

In [3]:
# includes all the folders of the different years
folders = [f"{path}/Stella1oAnno", f"{path}/Canal2oAnno", f"{path}/Lombardi3oAnno"]

# function to associate the name of the year to all the files based on their initial characters 
def year_extraction(filename):
    if filename.startswith('0'):
       return'first_year'
    if filename.startswith('LC'):
        return 'second_year'
    if filename.startswith('LL'):
        return 'third_year'


# create a list that has the filename, its ordinal number, the year and that creates also the participant_id
filenames = []

for folder_path in folders:
    for filename in os.listdir(folder_path):
        year = year_extraction(filename)
        if filename.endswith(".xlsx"):
            match = re.search(r'(\d{3})', filename)
            if match:
                filenames.append({
                    'number': int(match.group(1)), 
                    'year' : year, 
                    'filename' : filename, 
                    'file_path': os.path.join(folder_path, filename),
                    'participant_id': f"{year}_{match.group(1)}",
                    })

# make the list into a df, sorted by year and number 
filenames = pd.DataFrame(filenames).sort_values(by = ['year', 'number']).reset_index(drop=True)

# takes only the 'participant_id' and stores it into a variable
participant_ids = filenames['participant_id']

In [4]:
# all the 41 keywords used in the experiment
keywords = ['voto', 'compito', 'noia', 'stress', 'biologia', 'comportamento', 'scoperta', 'università', 'neuroscienze', 'grafico', 'cognizione', 'emozione', 'sfida', 'creatività', 'scienza', 'statistica', 'relazione', 'lavoro', 'tecnologia', 'equazione', 'fisica', 'psicologia','personalità', 'terapia', 'esercitazione', 'mente', 'numeri', 'futuro', 'frustrazione', 'matematica', 'esame', 'benessere', 'modello',  'professore', 'ansia', 'laurea', 'conoscenza', 'curiosità', 'scuola', 'insegnante', 'studente']

# function to iterate over the files, read them and store them into a df
def file_iteration(folders, sheet_name):
    for file_path, filename in zip(filenames['file_path'], filenames['filename']):
        df = pd.read_excel(file_path, sheet_name=sheet_name)
        yield df, filename

# function to check which keyword is missing from the file, describe the findings and store them in an excel file 
def missing_cue_words(df, keywords, filename, save_path):

    missing_words = []
    
    df['cue word'] = df['cue word'].str.strip() 
    if len(df) != 41:
        missing_keywords = [keyword for keyword in keywords if keyword not in df['cue word'].values]
        missing_words.append(f'filename: {filename}, length: {len(df)}, missing words: {missing_keywords}')

    with open(save_path, "a") as file:
        for line in missing_words:
            file.write(line + "\n")

In [5]:
# function to create lists 
def build_lists(df, filename, dict_lists, col_names):

    # loop to create lists and append them to the dictionary
    for col_name in col_names:
        dict_lists[col_name].append(df[col_name].reset_index(drop=True))
    
    # since there are missing words, let's get the length of the df (number of cue words)
    num_rows = len(df)

    # 
    participant_id = dict(zip(filenames['filename'], filenames['participant_id'])).get(filename)

    # append the participant_id to the dictionary multiplying it by the number of cue words present in the df
    dict_lists['participant_id'].append(pd.Series([participant_id] * num_rows))
    
    # creates a df for lists
    combined_lists = {
    key: pd.concat(series_list, ignore_index=True)
    for key, series_list in dict_lists.items()
    }

    # sort the df by 'cue word' at first and then by 'filename'
    lists_df = pd.DataFrame(combined_lists).sort_values(by=['cue word', 'participant_id']).reset_index(drop=True)

    # rename 'cue word' to 'cue_word'
    lists_df.rename(columns={"cue word": "cue_word"}, inplace=True)

    # insert as first column the 'filename' and then all the other columns
    cols = ['participant_id'] + [col for col in lists_df.columns if col != 'participant_id']
    lists_df = lists_df[cols]

    # save associations lists to .csv
    # lists_df.to_csv(file_path, index=False)

    # return the df
    return lists_df

In [6]:
# function to create series 
def build_series(df, map_dict, col_names, file_path):

    # loop to create series mapping over the dictionary associations_map
    for _, row in df.iterrows():
        cue = row["cue word"].strip()
        for col in col_names:
            if col != "cue word":
                map_dict[cue][col].append(row[col])

    # turns into a df with lists in each cell
    series_df = pd.DataFrame([
        {
            'cue word': cue,
            **{f'all_{col}': pd.Series(data[col]) for col in col_names if col != 'cue word'}
        }
        for cue, data in map_dict.items()
    ])

    # sorts the df by cue word
    series_df = series_df.sort_values(by='cue word').reset_index(drop=True)

    # saves the df into a csv file
    series_df.to_csv(file_path, index=False)

# Task associazioni

In [8]:
output_path = "/Users/emma/projects/cognoscolab/rassereno/out/formatting/psychology_BA_2024"

In [9]:
# names of the columns in the association task
col_names_associations = ["cue word", "association_1", "association_2", "association_3"]

# creates a dictionary for each column, except for 'cue word', used as index
associations_map = defaultdict(
    lambda: {col: [] for col in col_names_associations if col != "cue word"}
)

# created a dictionary of lists to store the association, it creates a default value for keys that don't exist yet
associations_lists = defaultdict(list)

# save path for the .txt file of the missing keywords
save_path = f"{output_path}/missing_keywords_associations_task.txt"

# closes the file so that it writes the new information
open(save_path, "w").close()

# call of iteration function of the associations task sheet
for df, filename in file_iteration(folders, "Task_1parte_Associazioni"):

    df = df[col_names_associations][10:].dropna(how='all')
    
    # call the function to create lists and save them in csv file
    associations_df = build_lists(df, filename, associations_lists, col_names_associations)

    # call of function missing keywords in the association task
    missing_cue_words(df, keywords, filename, save_path)

    # call the function to create lists of series and save them in csv file
    # build_series(df, associations_map, col_names_associations, 'extracted_data_2024/associations_task_2024_series.csv')

In [10]:
problem_files = [
    'first_year_011',
    'first_year_012',
    'first_year_013',
    'first_year_015',
    'first_year_022',
    'first_year_026',
    'first_year_037',
    'first_year_039',
    'first_year_040',
]

In [11]:
excluded_mask = (associations_df['participant_id'].isin(problem_files)) & (associations_df['cue_word'].isin(['biologia', 'psicologia']))

associations_df_trunked = associations_df[~excluded_mask]

associations_df_trunked.shape

(6822, 5)

# Task valenza

In [12]:
# names of the columns for the valence task 
col_names_valence = ['cue word', 'valence_cue_word', 'association_1', 'valence_association_1', 'association_2', 'valence_association_2', 'association_3', 'valence_association_3']

valence_scores_map = defaultdict(
    lambda: {col: [] for col in col_names_valence if col != "cue word"}
)

# created a dictionary of lists to store the associations and valence scores, it creates a default value for keys that don't exists yet
valence_lists = defaultdict(list)

# save path for the .txt file of the missing keywords
save_path = f'{output_path}/missing_keywords_valence_task.txt'

# closes the file so that it writes the new information
open(save_path, "w").close()

# call of iteration function of the associations task sheet
for df, filename in file_iteration(folders, 'Task_2parte_Valenza'):

    # takes the values, the data concerning the valence scores
    df = df.iloc[8:, range(8)].dropna(how='all')
    
    # assigns the column names to the columns 
    df.columns = col_names_valence

    # call the function to create lists and save them in csv file
    valence_df = build_lists(df, filename, valence_lists, col_names_valence)

    # call the function to create lists of series and save them in csv file
    # build_series(df, valence_scores_map, col_names_valence, 'extracted_data_2024/valence_task_2024_series.csv')
    
    # call of function missing keywords in the valence task
    missing_cue_words(df, keywords, filename, save_path)

In [13]:
valence_columns = [
    'valence_cue_word',
    'valence_association_1',
    'valence_association_2',
    'valence_association_3'
]
for col in valence_columns:
    valence_df[col] = pd.to_numeric(valence_df[col], errors='coerce')

for col in valence_df.columns:
    if valence_df[col].dtype == object:
        valence_df[col] = valence_df[col].str.strip()

valence_df.replace([int(0)], np.nan)

,participant_id,cue_word,valence_cue_word,association_1,valence_association_1,association_2,valence_association_2,association_3,valence_association_3
0,first_year_001,ansia,1.0,Stress,1.0,Trasferimento,1.0,Sociale,3.0
1,first_year_002,ansia,1.0,notte,3.0,vita,4.0,scuola,2.0
2,first_year_003,ansia,1.0,io,3.0,disturbo,2.0,casino,2.0
3,first_year_004,ansia,1.0,aspettative,2.0,no controllo,1.0,paura,1.0
4,first_year_005,ansia,1.0,noia,1.0,passato,5.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...
6817,third_year_048,voto,1.0,ansia,2.0,definizione,4.0,numero,4.0
6818,third_year_049,voto,3.0,alto,5.0,frustrazione,3.0,giudizio,3.0
6819,third_year_050,voto,3.0,numero,3.0,valutazione,3.0,classe,3.0
6820,third_year_051,voto,1.0,Media,1.0,Importanza,1.0,Fastidio,1.0


# Merge association_df and valence_df

In [14]:
def compare_df(df1, df2):
    if df1.equals(df2) == False:
        comparison = (df1 == df2) | (df1.isna() & df2.isna())

        mismatched_mask = ~comparison.all(axis=1)
        mismatched_rows_df1 = df1[mismatched_mask]
        mismatched_rows_df2 = df2[mismatched_mask]

        with open('differences.txt', "w") as file:
            for (_, row_df1), (_, row_df2) in zip(mismatched_rows_df1.iterrows(), mismatched_rows_df2.iterrows()):
                file.write(str(row_df1.to_dict()) + "\n" + str(row_df2.to_dict()) + "\n")
        return 'False: the two dataframes are different'
    else:
        return 'True: the two dataframes are equal'

In [15]:
columns_to_compare = ["participant_id", "cue_word", "association_1", "association_2", "association_3"]

df1_subset = associations_df_trunked[columns_to_compare].applymap(lambda x: x.strip() if isinstance(x, str) else x)
df2_subset = valence_df[columns_to_compare].applymap(lambda x: x.strip() if isinstance(x, str) else x)

df1_subset = df1_subset.reset_index(drop=True)
df2_subset = df2_subset.reset_index(drop=True)

df1_subset = df1_subset.combine_first(df2_subset)
df2_subset = df2_subset.combine_first(df1_subset)

compare_df(df1_subset, df2_subset)

/var/folders/nt/hqv745w1077gw1x2h5r4m2p00000gn/T/ipykernel_71028/3608650500.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df1_subset = associations_df_trunked[columns_to_compare].applymap(lambda x: x.strip() if isinstance(x, str) else x)
/var/folders/nt/hqv745w1077gw1x2h5r4m2p00000gn/T/ipykernel_71028/3608650500.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df2_subset = valence_df[columns_to_compare].applymap(lambda x: x.strip() if isinstance(x, str) else x)


'True: the two dataframes are equal'

In [16]:
df2_subset_clean = df2_subset.applymap(lambda x: x.strip() if isinstance(x, str) else x)

/var/folders/nt/hqv745w1077gw1x2h5r4m2p00000gn/T/ipykernel_71028/3820327731.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df2_subset_clean = df2_subset.applymap(lambda x: x.strip() if isinstance(x, str) else x)


In [17]:
# Get columns that were excluded from the subset
excluded_columns = [col for col in valence_df.columns if col not in columns_to_compare]

# Merge the original valence_df with the cleaned subset to get full records
df_cleaned = df2_subset_clean.merge(
    valence_df[columns_to_compare + excluded_columns],
    # on=columns_to_compare,
    how='right',
    validate='one_to_one' 
)

col_order = ['participant_id', 'cue_word', 'valence_cue_word', 'association_1', 'valence_association_1', 'association_2', 'valence_association_2', 'association_3', 'valence_association_3']

df_cleaned = df_cleaned[col_order]

In [18]:
excluded_rows = associations_df[excluded_mask].copy()

for col in df_cleaned.columns:
    if col not in excluded_rows.columns:
        excluded_rows[col] = np.nan

excluded_rows = excluded_rows[df_cleaned.columns]
merged_df = pd.concat([df_cleaned, excluded_rows[col_order]]).sort_values(by=['cue_word', 'participant_id']).reset_index(drop=True)

merged_df.to_csv(f'{output_path}/associations_valence_2024.csv', index=False)

In [19]:
expected_nan_values = valence_df[valence_df[valence_columns].isna().all(axis=1)]
unexpected_nan_values = df_cleaned[df_cleaned[valence_columns].isna().all(axis=1)]

compare_df(unexpected_nan_values, expected_nan_values)

'True: the two dataframes are equal'

In [20]:
final_nan_values = merged_df[merged_df[valence_columns].isna().all(axis=1)]
final_nan_values.shape 
# it should be 86 --> 68 (initial NaN values of the valence_df) + 18 (NaN values of the associations words present in the associations_df but not in the valence_df)

(49, 9)

# Parte dati demografici e questionari

## Dati demografici

In [21]:
# creates the lists for gender and age and assigns to them the values corresponding to certain excel cells
age = []
gender = []

for df, filename in file_iteration(folders, 'Dati_Demografici'):
    age.append([float(df.iloc[5, 1])])
    gender.append([df.iloc[9, 1]])

# transform them into df with a column name
age_df = pd.DataFrame(age, columns=['age'])
gender_df = pd.DataFrame(gender, columns=['gender'])

/Users/emma/projects/cognoscolab/myvenv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


## Questionari

In [22]:
# function to parse the values found in the excel cells
def parse_value(x):
    # check if the values are NaN
    if pd.isna(x):
        return None
    try:
        # if it's already a number (int or float) return it directly
        if isinstance(x, (int, float)):
            return int(x) if float(x).is_integer() else float(x)
        
        # if the value is a string we take the first part of the string and normalize it
        token = str(x).strip().split()[0]
        
        # converting the token of the the string to float only if it's a decimal, if not we transform it into a integer
        number = float(token)
        return int(number) if number.is_integer() else number
    except:
        return None

In [23]:
# function that allows to extract the data from the questionnaire 
def extract_questionnaire_data(folders, sheet_name, row_range, column_index, headers):
    # creates empty list to store the values found in a determined position in the excel file
    data = []

    # extracts the values and it cleans them from unnecessary text (for example in the likert scores descriptions) 
    for df, filename in file_iteration(folders, sheet_name):

        # extract the values from specific excel cells
        values = df.iloc[row_range, column_index].values
        
        # calls the parsing function to get the cleaned data from the cells values
        cleaned_values = [parse_value(x) for x in values]

        # append the cleaned data to the list 
        data.append(cleaned_values)
        
    # makes the list into a df with specific headers
    return pd.DataFrame(data, columns=headers)

In [24]:
# Math abilities

math_abilities = extract_questionnaire_data(
    folders = folders, 
    sheet_name='Questionari', 
    row_range= range(5, 12, 2), 
    column_index=1,
    headers = ['materie STEM', 'meterie umanistiche', 'matematica e fisica', 'programmazione']
)

/Users/emma/projects/cognoscolab/myvenv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [25]:
# Need for cognition

need_cognition = extract_questionnaire_data(
    folders=folders,
    sheet_name='Questionari',
    row_range=range(20, 56, 2),
    column_index=1,
    headers=[f'need_cognition_{i}' for i in range(1, 19)]
)

/Users/emma/projects/cognoscolab/myvenv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [26]:
# Big 5 reduced

big_5 = extract_questionnaire_data(
    folders=folders,
    sheet_name='Questionari',
    row_range=range(64, 84, 2),
    column_index=1,
    headers=[f'big_5_{i}' for i in range(1, 11)]
)

/Users/emma/projects/cognoscolab/myvenv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [29]:
# MAS-IT

mas_it = extract_questionnaire_data(
    folders=folders,
    sheet_name='Questionari',
    row_range=range(91, 137, 2),
    column_index=1,
    headers=[f'mas_it_{i}' for i in range(1, 24)]
)

correspondence_order = [
    'mas_it_1', 'mas_it_3', 'mas_it_6', 'mas_it_7', 'mas_it_10', 'mas_it_18', 'mas_it_19',
    'mas_it_21', 'mas_it_23', 'mas_it_2', 'mas_it_4', 'mas_it_5', 'mas_it_8', 'mas_it_11',
    'mas_it_13', 'mas_it_14', 'mas_it_22', 'mas_it_9', 'mas_it_12', 'mas_it_15',
    'mas_it_16', 'mas_it_17', 'mas_it_20'
]

# reorder columns based on the original order
mas_it_reordered = mas_it[correspondence_order]

# # name columns to match the original names
mas_it_reordered.columns = mas_it.columns

/Users/emma/projects/cognoscolab/myvenv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [32]:
mas_it_df = pd.concat([participant_ids, mas_it_reordered], axis=1)
mas_it_df.to_csv(f'{output_path}/mas_it_2024.csv', index=False)

In [35]:
# STAI: State Anxiety

state_anxiety = extract_questionnaire_data(
    folders=folders,
    sheet_name='Questionari',
    row_range=range(143, 157, 2),
    column_index=1,
    headers=[f'state_anxiety_{i}' for i in range(1, 8)]
)

/Users/emma/projects/cognoscolab/myvenv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


## Merge and export to .csv all the dfs

In [36]:
# concatenates all the df related to demographic data and questionnaire response + the participants ids
all_df = pd.concat([participant_ids, age_df, gender_df, math_abilities, need_cognition, big_5, mas_it, state_anxiety], axis=1)

# exports the concatenated dfs into an csv file
all_df.to_csv(f'{output_path}/all_questionnaires_task_2024.csv', index=False)

# Breve testo

In [32]:
# these are the strings corresponding to the instructions given by the researchers
instruction_1 = 'Scriva adesso un breve testo di circa 250 parole relativamente a come si sente quando si trova ad affrontare le discipline STEM.'
instruction_2 = '"Quando affronto le discipline STEM…"'
# unnecessary string
punctuation = '…'

# creates a list with all the texts of all the participant extracted in the excel files
text = []

for df, filename in file_iteration(folders, 'breve_testo'):
    df = df.dropna().reset_index(drop=True)
    participant_text = " ".join(df.iloc[:, 0].astype(str))
    # creates a string with '.' at the end and removes alla the instruction strings
    participant_text = f"{participant_text.replace(instruction_1, '').replace(instruction_2, '').replace(punctuation, '').strip()}."
    # appends the text for each participant in the list
    text.append(participant_text)

# converts the text into a df 
texts = pd.DataFrame(text, columns=['text'])
# concatenates the text and the participants ids
texts_df = pd.concat([participant_ids, texts], axis=1)
# export the df into a csv file
texts_df.to_csv(f'{output_path}/text_task_2024.csv', index=False)
texts_df

,participant_id,text
0,first_year_001,.
1,first_year_002,.
2,first_year_003,Quando affronto le discipline STEM ammetto di ...
3,first_year_004,.
4,first_year_005,"Mi sento più curioso che spaventato, spero che..."
...,...,...
163,third_year_048,Quando affronto le discipline STEM non mi sent...
164,third_year_049,Mi sento abbastanza traquillo anche se non sem...
165,third_year_050,mi sento molto invogliata. Mi interessano part...
166,third_year_051,Quando affondo le discipline STEM mi sento mol...


In [30]:
# merges the content of the text of all the participants into a single text
merged_text = ".".join(texts_df['text'].apply(lambda x: re.sub(r'(?i)quando affronto le discipline STEM', '', x)))
len(merged_text)

90064